# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIRˆ² dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library. 

The dataset covers ordered logistic regression results for household knowledge adoption in rangeland management (Northern Kenya).

### Dataset Source
The dataset source is provided via a [Croissant schema JSON-LD](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Identifier: {getattr(meta, 'identifier', '[none]')}")
print(f"License: {getattr(meta, 'license', '[none]')}")
print(f"Location: {getattr(meta, 'spatialCoverage', '[none]')}")

## 2. Data Overview

Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets in the dataset by their @id, then enumerate their fields (if possible)

print("Available record sets and their @id values:")

record_sets = [rs for rs in dataset.record_sets()]
for i, record_set in enumerate(record_sets):
    fields = record_set.fields
    print(f"[{i+1}] RecordSet @id: {record_set.id}")
    print(f"    Name: {getattr(record_set, 'name', '[no name]')}")
    print(f"    Description: {getattr(record_set, 'description', '[no description]')}")
    print(f"    Fields:")
    for field in fields:
        print(f"      - Field @id: {field.id} | name: {getattr(field, 'name', '[no name]')} | dataType: {getattr(field, 'data_type', '[none]')}")
    print()

## 3. Data Extraction
Load data from available record sets into Pandas DataFrames for analysis. Use the record set and field `@id`s discovered above.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}
for record_set in record_set_ids:
    # Records yielded as dicts with field @id as key
    recs = list(dataset.records(record_set=record_set))
    dataframes[record_set] = pd.DataFrame(recs)

for record_set in record_set_ids:
    print(f"RecordSet {record_set} columns:")
    print(dataframes[record_set].columns.tolist())
    display(dataframes[record_set].head(3))

## 4. Exploratory Data Analysis (EDA)
This section demonstrates: filtering records, normalizing numeric fields, grouping by categorical fields, and previewing the processed data. All field and record set references are by their `@id`.

In [ ]:
# Example: Pick the first record set for demo
if len(record_set_ids) > 0:
    rec_id = record_set_ids[0]
    df = dataframes[rec_id]
    print(f"Using RecordSet @id: {rec_id} for analysis.")
    # Find first numeric field by '@id', fallback to any field
    example_field = None
    for field in dataset.record_set(rec_id).fields:
        if getattr(field, 'data_type', None) in ('Float', 'Integer', 'Number', 'schema:Float', 'schema:Integer', 'schema:Number'):
            example_field = field.id
            break
    if not example_field:
        # Just choose any if none found
        example_field = df.columns[0]
    print(f"Numeric field selected by @id: {example_field}")
    # Try to convert numeric, ignore errors
    df = df.copy()
    df[example_field] = pd.to_numeric(df[example_field], errors='coerce')
    # Drop NA to work with valid numerics
    thresh = 10
    filtered_df = df[df[example_field] > thresh]
    print(f"Filtered records where {example_field} > {thresh}:")
    print(filtered_df.head())
    # Normalize the numeric field
    norm_col = f"{example_field}_normalized"
    filtered_df[norm_col] = (filtered_df[example_field] - filtered_df[example_field].mean()) / filtered_df[example_field].std()
    print(f"Normalized {example_field} for filtered records:")
    print(filtered_df[[example_field, norm_col]].head())
    # Attempt grouping by another field (first string/categorical field found)
    group_field = None
    for field in dataset.record_set(rec_id).fields:
        if getattr(field, 'data_type', '').lower() in ('text', 'string', 'schema:text') and field.id != example_field:
            group_field = field.id
            break
    if group_field and group_field in filtered_df.columns:
        grouped = filtered_df.groupby(group_field)[example_field].mean().reset_index()
        print(f"Grouped mean of {example_field} by {group_field}:")
        print(grouped.head())
else:
    print("No record sets found in the dataset.")

## 5. Visualization
Visualize numeric distributions or categorical group comparisons for the selected record set.

In [ ]:
import matplotlib.pyplot as plt

if len(record_set_ids) > 0 and example_field in df.columns:
    plt.figure(figsize=(8,4))
    df[example_field].dropna().hist(bins=25)
    plt.xlabel(example_field)
    plt.ylabel("Frequency")
    plt.title(f"Histogram of {example_field}")
    plt.show()
    # Categorical comparison plot if grouping available
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10,4))
        df.boxplot(column=example_field, by=group_field, vert=False)
        plt.title(f"{example_field} by {group_field}")
        plt.ylabel(group_field)
        plt.xlabel(example_field)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

- This notebook illustrated loading and exploring the FAIRˆ² dataset using `mlcroissant` with references by `@id` for all entities.
- After loading the dataset, we listed all record sets, fields, and extracted data using only entity `@id`.
- Basic exploratory data operations and visualizations were demonstrated; further analysis can be tailored to research needs using the detailed dataset schema and precise entity IDs.

For more advanced analysis and FAIR practices, refer to additional Croissant/MLCommons documentation.